In [3]:
import sys
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt, colors, patches

from sklearn.cross_decomposition import CCA


sys.path.append(os.path.join(os.getcwd(), '../../'))
from utils import read_utils
from utils import plot_utils
from utils import train_utils

In [ ]:
import importlib
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)

In [ ]:
selection_n = ['DD01', 'VB02', 'SMDDR', 'AVAR']
selection_bhl = ['REV', 'VENTRAL_TURN', 'DORSAL_TURN', 'PAUSE']

dims = 3
dims_preca = 5

plot_id = 19

In [ ]:
path = os.path.join(read_utils.get_data_path(), 'worm-2025-fieseler', 'exported_data_gcamp')
(file_ids, filenames,
 traces,
 continuous_behaviors,
 discrete_states, eigenworms,
 kymographs) = read_utils.get_data(path)

# clear nan values
for i in range(len(file_ids)):
	mask = eigenworms[i].notna().all(axis=1)

	traces[i] = traces[i][mask]
	#continuous_behaviors[i] = continuous_behaviors[i][mask]
	eigenworms[i] = eigenworms[i][mask]
	discrete_states[i] = discrete_states[i][mask]

	#kymographs[i] = kymographs[i][mask]
	#filenames[i] = filenames[i][mask]

# remove unused discrete state labels
# discrete_states = [ds[selection_bhl] for ds in discrete_states]

In [ ]:
(modes_manifold, modes_manifold_b,
 recos_manifold, recos_manifold_b,
 loads_cca_man, loads_cca_man_b,
 vars_expl_mans, vars_expl_mans_rs) = train_utils.get_manifolds_cca(traces,
																	[ds[selection_bhl] for ds in discrete_states],
																	n_cv=dims, n_pc=dims_preca)

In [ ]:
# plot_utils.plot_manifolds([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
# 						  modes_manifold, [ds[['FWD'] + selection_bhl] for ds in discrete_states],
# 						  [vx[:2] for vx in vars_expl_mans], [vx[:2] for vx in vars_expl_mans_rs] if dims_preca > 0 else None,
# 						  title='CCA manifolds', save_name=None)

In [ ]:
traces_residuals = [pd.DataFrame(data=t.values - t_man.values, columns=t.columns) for t, t_man in zip(traces, recos_manifold)]

In [ ]:
n_cv = eigenworms[file_ids.index(plot_id)].values.shape[1]

modes_concrete, modes_concrete_b = list(), list()
recos_concrete, recos_concrete_b = list(), list()
loads_cca_con, loads_cca_con_b = list(), list()
vars_expl_con, vars_expl_con_rs = list(), list()  # rs in this case: variance explained with respect to the manifold-residual data
for trace_fwd, trace_residual_fwd, ew_fwd in zip(traces, traces_residuals, eigenworms):
	# reduction
	cca = CCA(n_components=n_cv, scale=True)

	trace_residual_reduced, ew_reduced = cca.fit_transform(trace_residual_fwd.values, ew_fwd.values)
	trace_residual_reconstructed, ew_reconstructed = cca.inverse_transform(trace_residual_reduced, ew_reduced)

	modes_concrete.append(pd.DataFrame(data=trace_residual_reduced, columns=[f'CC{i + 1}' for i in range(n_cv)]))
	recos_concrete.append(pd.DataFrame(data=trace_residual_reconstructed, columns=trace_residual_fwd.columns))

	loads_cca_con.append(cca.x_loadings_)
	loads_cca_con_b.append(cca.y_loadings_)

	# variance explained
	var_ratio = list()
	var_ratio_rs = list()

	var_ratio_cumulative = list()
	var_ratio_rs_cumulative = list()
	for k in range(1, n_cv+1):  # VE k = VE :k - VE :k-1
		trace_residual_reduced_to_k = np.zeros_like(trace_residual_reduced)
		trace_residual_reduced_to_k[:, :k] = trace_residual_reduced[:, :k]
		ew_reduced_to_k = np.zeros_like(ew_reduced)
		ew_reduced_to_k[:, :k] = ew_reduced[:, :k]

		trace_reconstructed_to_k, _ = cca.inverse_transform(trace_residual_reduced_to_k, ew_reduced_to_k)

		var_ratio_cumulative.append(train_utils.get_variance_explained(trace_reconstructed_to_k, trace_fwd))
		var_ratio_rs_cumulative.append(train_utils.get_variance_explained(trace_reconstructed_to_k, trace_residual_fwd))

		var_ratio_k = var_ratio_cumulative[k-1] - (var_ratio_cumulative[k-1-1] if k > 1 else 0)
		var_ratio_rs_k = var_ratio_rs_cumulative[k-1] - (var_ratio_rs_cumulative[k-1-1] if k > 1 else 0)

		var_ratio.append(var_ratio_k)
		var_ratio_rs.append(var_ratio_rs_k)

	vars_expl_con.append(var_ratio)
	vars_expl_con_rs.append(var_ratio_rs)

In [ ]:
plot_utils.plot_manifolds([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
						  modes_concrete, eigenworms,
						  [vx[:2] for vx in vars_expl_con], [vx[:2] for vx in vars_expl_con_rs],
						  title='CCA manifolds', save_name=None)

In [ ]:
plot_utils.plot_manifolds_3d([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
							 modes_concrete, eigenworms,
							 [vx[:3] for vx in vars_expl_con], [vx[:3] for vx in vars_expl_con_rs],
							 title='CCA manifolds', save_name=None)

In [ ]:
plot_utils.plot_manifolds_3d_sep([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
								 modes_concrete, eigenworms,
								 [vx[:3] for vx in vars_expl_con], [vx[:3] for vx in vars_expl_con_rs],
								 animate=True, n_plots=1,
								 titles=['CCA manifolds'] * len(file_ids), save_name=f'{f'{dims_preca}-dim-PCA_' if dims_preca > 0 else ''}{dims}-dim-ConCCA_3d')

In [ ]:
fwd_indices = [np.where(ds['FWD'].values == 1) for ds in discrete_states]
rev_indices = [np.where(ds['REV'].values == 1) for ds in discrete_states]

traces_fwd = [pd.DataFrame(data=trace.values[fwd_index], columns=trace.columns) for trace, fwd_index in zip(traces, fwd_indices)]
traces_rev = [pd.DataFrame(data=trace.values[rev_index], columns=trace.columns) for trace, rev_index in zip(traces, rev_indices)]

traces_residuals_fwd = [pd.DataFrame(data=trace.values[fwd_index], columns=trace.columns) for trace, fwd_index in zip(traces_residuals, fwd_indices)]
traces_residuals_rev = [pd.DataFrame(data=trace.values[rev_index], columns=trace.columns) for trace, rev_index in zip(traces_residuals, rev_indices)]

eigenworms_fwd = [pd.DataFrame(data=ew.values[fwd_index], columns=ew.columns) for ew, fwd_index in zip(eigenworms, fwd_indices)]
eigenworms_rev = [pd.DataFrame(data=ew.values[rev_index], columns=ew.columns) for ew, rev_index in zip(eigenworms, rev_indices)]

discrete_states_fwd = [pd.DataFrame(data=ds.values[fwd_index], columns=ds.columns) for ds, fwd_index in zip(discrete_states, fwd_indices)]
discrete_states_rev = [pd.DataFrame(data=ds.values[rev_index], columns=ds.columns) for ds, rev_index in zip(discrete_states, rev_indices)]

In [ ]:
n_cv = eigenworms_fwd[file_ids.index(plot_id)].values.shape[1]

modes_concrete_fwd, modes_concrete_fwd_b = list(), list()
recos_concrete_fwd, recos_concrete_fwd_b = list(), list()
loads_cca_con_fwd, loads_cca_con_fwd_b = list(), list()
vars_expl_con_fwd, vars_expl_con_fwd_rs = list(), list()  # rs in this case: variance explained with respect to the manifold-residual data
for trace_fwd, trace_residual_fwd, ew_fwd in zip(traces_fwd, traces_residuals_fwd, eigenworms_fwd):
	# reduction
	cca = CCA(n_components=n_cv, scale=True)

	trace_residual_reduced, ew_reduced = cca.fit_transform(trace_residual_fwd.values, ew_fwd.values)
	trace_residual_reconstructed, ew_reconstructed = cca.inverse_transform(trace_residual_reduced, ew_reduced)

	modes_concrete_fwd.append(pd.DataFrame(data=trace_residual_reduced, columns=[f'CC{i + 1}' for i in range(n_cv)]))
	recos_concrete_fwd.append(pd.DataFrame(data=trace_residual_reconstructed, columns=trace_residual_fwd.columns))

	loads_cca_con_fwd.append(cca.x_loadings_)
	loads_cca_con_fwd_b.append(cca.y_loadings_)

	# variance explained
	var_ratio = list()
	var_ratio_rs = list()

	var_ratio_cumulative = list()
	var_ratio_rs_cumulative = list()
	for k in range(1, n_cv+1):  # VE k = VE :k - VE :k-1
		trace_residual_reduced_to_k = np.zeros_like(trace_residual_reduced)
		trace_residual_reduced_to_k[:, :k] = trace_residual_reduced[:, :k]
		ew_reduced_to_k = np.zeros_like(ew_reduced)
		ew_reduced_to_k[:, :k] = ew_reduced[:, :k]

		trace_residual_reconstructed_to_k, _ = cca.inverse_transform(trace_residual_reduced_to_k, ew_reduced_to_k)

		var_ratio_cumulative.append(train_utils.get_variance_explained(trace_residual_reconstructed_to_k, trace_fwd))
		var_ratio_rs_cumulative.append(train_utils.get_variance_explained(trace_residual_reconstructed_to_k, trace_residual_fwd))

		var_ratio_k = var_ratio_cumulative[k-1] - (var_ratio_cumulative[k-1-1] if k > 1 else 0)
		var_ratio_rs_k = var_ratio_rs_cumulative[k-1] - (var_ratio_rs_cumulative[k-1-1] if k > 1 else 0)

		var_ratio.append(var_ratio_k)
		var_ratio_rs.append(var_ratio_rs_k)

	vars_expl_con_fwd.append(var_ratio)
	vars_expl_con_fwd_rs.append(var_ratio_rs)

In [ ]:
plot_utils.plot_manifolds([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
						  modes_concrete_fwd, eigenworms_fwd,
						  [vx[:2] for vx in vars_expl_con_fwd], [vx[:2] for vx in vars_expl_con_fwd_rs],
						  title='CCA manifolds FWD', save_name=None)

In [ ]:
plot_utils.plot_manifolds_3d([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
							 modes_concrete_fwd, eigenworms_fwd,
							 [vx[:3] for vx in vars_expl_con_fwd], [vx[:3] for vx in vars_expl_con_fwd_rs],
							 title='CCA manifolds FWD', save_name=None)

In [ ]:
plot_utils.plot_manifolds_3d_sep([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
								 modes_concrete_fwd, eigenworms_fwd,
								 [vx[:3] for vx in vars_expl_con_fwd], [vx[:3] for vx in vars_expl_con_fwd_rs],
								 animate=True, n_plots=1,
								 titles=['CCA manifolds FWD'] * len(file_ids), save_name=f'{f'{dims_preca}-dim-PCA_' if dims_preca > 0 else ''}{dims}-dim-ConCCA_FWD_3d')

In [ ]:
n_cv = eigenworms_rev[file_ids.index(plot_id)].values.shape[1]

modes_concrete_rev, modes_concrete_rev_b = list(), list()
recos_concrete_rev, recos_concrete_rev_b = list(), list()
loads_cca_con_rev, loads_cca_con_rev_b = list(), list()
vars_expl_con_rev, vars_expl_con_rev_rs = list(), list()  # rs in this case: variance explained with respect to the manifold-residual data
for trace_rev, trace_residual_rev, ew_rev in zip(traces_rev, traces_residuals_rev, eigenworms_rev):
	# reduction
	cca = CCA(n_components=n_cv, scale=True)

	trace_residual_reduced, ew_reduced = cca.fit_transform(trace_residual_rev.values, ew_rev.values)
	trace_residual_reconstructed, ew_reconstructed = cca.inverse_transform(trace_residual_reduced, ew_reduced)

	modes_concrete_rev.append(pd.DataFrame(data=trace_residual_reduced, columns=[f'CC{i + 1}' for i in range(n_cv)]))
	recos_concrete_rev.append(pd.DataFrame(data=trace_residual_reconstructed, columns=trace_residual_rev.columns))

	loads_cca_con_rev.append(cca.x_loadings_)
	loads_cca_con_rev_b.append(cca.y_loadings_)

	# variance explained
	var_ratio = list()
	var_ratio_rs = list()

	var_ratio_cumulative = list()
	var_ratio_rs_cumulative = list()
	for k in range(1, n_cv+1):  # VE k = VE :k - VE :k-1
		trace_residual_reduced_to_k = np.zeros_like(trace_residual_reduced)
		trace_residual_reduced_to_k[:, :k] = trace_residual_reduced[:, :k]
		ew_reduced_to_k = np.zeros_like(ew_reduced)
		ew_reduced_to_k[:, :k] = ew_reduced[:, :k]

		trace_residual_reconstructed_to_k, _ = cca.inverse_transform(trace_residual_reduced_to_k, ew_reduced_to_k)

		var_ratio_cumulative.append(train_utils.get_variance_explained(trace_residual_reconstructed_to_k, trace_rev))
		var_ratio_rs_cumulative.append(train_utils.get_variance_explained(trace_residual_reconstructed_to_k, trace_residual_rev))

		var_ratio_k = var_ratio_cumulative[k-1] - (var_ratio_cumulative[k-1-1] if k > 1 else 0)
		var_ratio_rs_k = var_ratio_rs_cumulative[k-1] - (var_ratio_rs_cumulative[k-1-1] if k > 1 else 0)

		var_ratio.append(var_ratio_k)
		var_ratio_rs.append(var_ratio_rs_k)

	vars_expl_con_rev.append(var_ratio)
	vars_expl_con_rev_rs.append(var_ratio_rs)

In [ ]:
plot_utils.plot_manifolds([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
						  modes_concrete_rev, eigenworms_rev,
						  [vx[:2] for vx in vars_expl_con_rev], [vx[:2] for vx in vars_expl_con_rev_rs],
						  title='CCA manifolds REV', save_name=None)

In [ ]:
plot_utils.plot_manifolds_3d([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
							 modes_concrete_rev, eigenworms_rev,
							 [vx[:3] for vx in vars_expl_con_rev], [vx[:3] for vx in vars_expl_con_rev_rs],
							 title='CCA manifolds REV', save_name=None)

In [ ]:
plot_utils.plot_manifolds_3d_sep([f'{str(file_id)}: {filenames[file_ids.index(file_id)]}' for file_id in file_ids],
								 modes_concrete_rev, eigenworms_rev,
								 [vx[:3] for vx in vars_expl_con_rev], [vx[:3] for vx in vars_expl_con_rev_rs],
								 animate=True, n_plots=1,
								 titles=['CCA manifolds REV'] * len(file_ids), save_name=f'{f'{dims_preca}-dim-PCA_' if dims_preca > 0 else ''}{dims}-dim-ConCCA_REV_3d')